<a href="https://colab.research.google.com/github/trang1981/ELAPS/blob/main/VIB_60DE1C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# E1 — NHÁNH C
# FLDC-60D + tenure_at_cutoff
#
# Nhãn 1:
#   mở thẻ sau ngày thứ 60 và không muộn hơn 31/12/2019
#
# Nhãn 0:
#   không mở thẻ đến 31/12/2019
#
# tenure_at_cutoff:
#   Nhãn 1 = first_card_date - relationship_date
#   Nhãn 0 = 31/12/2019 - relationship_date
#
# Code có thể chạy lại nhiều lần trong cùng phiên Colab.
# ============================================================


# ============================================================
# 0. IMPORT VÀ KẾT NỐI GOOGLE DRIVE
# ============================================================

from google.colab import drive

import os
import gc
import pandas as pd
import numpy as np

from pathlib import Path


# Chỉ mount nếu Google Drive chưa được mount
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Google Drive đã được mount. Tiếp tục sử dụng.")


# ============================================================
# 1. DỌN BIẾN CŨ TỪ LẦN CHẠY TRƯỚC
# ============================================================

variables_to_delete = [
    "final_df",
    "customer_df",
    "card_df",
    "customer_sub",
    "card_sub",
    "customer_dates",
    "positive_card_rows",
    "first_card_dates",
    "timeline_df",
    "branch_c_timeline",
    "branch_c_df",
    "early_adopter_df",
    "insufficient_window_df",
    "invalid_date_df",
    "positive_check",
    "negative_check",
    "tenure_summary",
    "label_statistics"
]

for variable_name in variables_to_delete:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

print("Đã dọn dữ liệu còn lại từ lần chạy trước.")


# ============================================================
# 2. TỰ ĐỘNG TÌM FILE NGUỒN
# ============================================================

DRIVE_ROOT = Path("/content/drive/MyDrive")


def find_source_file(filename):
    """
    Tìm file theo tên trong toàn bộ MyDrive.

    Nếu tìm thấy nhiều file trùng tên:
    - ưu tiên file không nằm trong Trash;
    - in danh sách;
    - chọn file đầu tiên.
    """

    matches = [
        path
        for path in DRIVE_ROOT.rglob(filename)
        if path.is_file()
        and ".Trash" not in str(path)
    ]

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Không tìm thấy file '{filename}' "
            f"trong thư mục {DRIVE_ROOT}."
        )

    matches = sorted(matches)

    if len(matches) > 1:
        print("\n" + "=" * 78)
        print(f"TÌM THẤY {len(matches)} FILE TRÙNG TÊN: {filename}")
        print("=" * 78)

        for i, path in enumerate(matches, start=1):
            print(f"{i}. {path}")

        print("\nMặc định sử dụng file đầu tiên:")
        print(matches[0])

    return matches[0]


FINAL_PATH = find_source_file(
    "final_dataset_no_auto_job.csv"
)

CUSTOMER_PATH = find_source_file(
    "1.Data_Customer.csv"
)

CARD_PATH = find_source_file(
    "6.Data_Card.xlsx"
)


# Lưu kết quả cùng thư mục với final dataset
BASE_DIR = FINAL_PATH.parent

OUTPUT_PATH = (
    BASE_DIR / "VIB_E1_BRANCH_C.csv"
)

TIMELINE_PATH = (
    BASE_DIR / "VIB_E1_BRANCH_C_timeline.csv"
)

EARLY_ADOPTER_PATH = (
    BASE_DIR
    / "VIB_E1_BRANCH_C_removed_early_adopters.csv"
)

INSUFFICIENT_WINDOW_PATH = (
    BASE_DIR
    / "VIB_E1_BRANCH_C_removed_insufficient_window.csv"
)

INVALID_DATE_PATH = (
    BASE_DIR
    / "VIB_E1_BRANCH_C_invalid_dates.csv"
)


print("\n" + "=" * 78)
print("ĐƯỜNG DẪN ĐƯỢC SỬ DỤNG")
print("=" * 78)

print("Final dataset :", FINAL_PATH)
print("Customer      :", CUSTOMER_PATH)
print("Card          :", CARD_PATH)
print("Output        :", OUTPUT_PATH)


# ============================================================
# 3. KHAI BÁO THAM SỐ
# ============================================================

HORIZON_DAYS = 60

FIXED_END_DATE = pd.Timestamp("2019-12-31")

ID_COL = "CUSTOMER_NUMBER"

RELATIONSHIP_DATE_COL = "CLIENT_CREATE_DATE"

CARD_MONTH_COL = "MONTH"

CARD_COUNT_COL = "COUNT_CREDITCARD"


# ============================================================
# 4. ĐỌC DỮ LIỆU
# ============================================================

final_df = pd.read_csv(
    FINAL_PATH,
    low_memory=False
)

customer_df = pd.read_csv(
    CUSTOMER_PATH,
    low_memory=False
)

card_df = pd.read_excel(
    CARD_PATH,
    engine="openpyxl"
)


# Chuẩn hóa tên cột
for df in [
    final_df,
    customer_df,
    card_df
]:
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )


print("\n" + "=" * 78)
print("KÍCH THƯỚC DỮ LIỆU BAN ĐẦU")
print("=" * 78)

print(f"Final dataset : {final_df.shape}")
print(f"Customer      : {customer_df.shape}")
print(f"Card          : {card_df.shape}")


# ============================================================
# 5. KIỂM TRA CỘT BẮT BUỘC
# ============================================================

required_columns = [
    (
        final_df,
        ID_COL,
        FINAL_PATH.name
    ),
    (
        customer_df,
        ID_COL,
        CUSTOMER_PATH.name
    ),
    (
        customer_df,
        RELATIONSHIP_DATE_COL,
        CUSTOMER_PATH.name
    ),
    (
        card_df,
        ID_COL,
        CARD_PATH.name
    ),
    (
        card_df,
        CARD_MONTH_COL,
        CARD_PATH.name
    ),
    (
        card_df,
        CARD_COUNT_COL,
        CARD_PATH.name
    )
]

for df, column_name, file_name in required_columns:
    if column_name not in df.columns:
        raise KeyError(
            f"Không tìm thấy cột '{column_name}' "
            f"trong file '{file_name}'.\n"
            f"Các cột hiện có:\n{df.columns.tolist()}"
        )

print("\nĐã xác định đầy đủ các cột bắt buộc.")


# ============================================================
# 6. CHUẨN HÓA MÃ KHÁCH HÀNG
# ============================================================

def normalize_customer_id(series):
    """
    Chuẩn hóa CUSTOMER_NUMBER giữa các file.
    """

    return (
        series
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "None": pd.NA,
            "<NA>": pd.NA
        })
    )


final_df["_CUSTOMER_ID"] = normalize_customer_id(
    final_df[ID_COL]
)

customer_df["_CUSTOMER_ID"] = normalize_customer_id(
    customer_df[ID_COL]
)

card_df["_CUSTOMER_ID"] = normalize_customer_id(
    card_df[ID_COL]
)


# ============================================================
# 7. CHUYỂN ĐỔI NGÀY VÀ GIÁ TRỊ SỐ
# ============================================================

customer_df["_RELATIONSHIP_DATE"] = pd.to_datetime(
    customer_df[RELATIONSHIP_DATE_COL],
    errors="coerce"
)

card_df["_CARD_MONTH"] = pd.to_datetime(
    card_df[CARD_MONTH_COL],
    errors="coerce"
)

card_df["_CARD_COUNT"] = pd.to_numeric(
    card_df[CARD_COUNT_COL],
    errors="coerce"
).fillna(0)


# ============================================================
# 8. KIỂM TRA THỜI GIAN
# ============================================================

card_min_date = card_df["_CARD_MONTH"].min()
card_max_date = card_df["_CARD_MONTH"].max()

if pd.isna(card_max_date):
    raise ValueError(
        "Không xác định được thời gian từ cột MONTH "
        "trong file Card."
    )


print("\n" + "=" * 78)
print("THÔNG TIN THỜI GIAN")
print("=" * 78)

print(f"Ngày nhỏ nhất trong Card : {card_min_date}")
print(f"Ngày lớn nhất trong Card : {card_max_date}")
print(f"Ngày kết thúc nghiên cứu : {FIXED_END_DATE.date()}")
print(f"Cửa sổ FLDC              : {HORIZON_DAYS} ngày")


# ============================================================
# 9. KIỂM TRA CUSTOMER_NUMBER TRONG FINAL DATASET
# ============================================================

n_missing_final_id = int(
    final_df["_CUSTOMER_ID"].isna().sum()
)

if n_missing_final_id > 0:
    raise ValueError(
        f"Final dataset có {n_missing_final_id:,} dòng "
        "thiếu CUSTOMER_NUMBER."
    )


if final_df["_CUSTOMER_ID"].duplicated().any():

    duplicated_ids = (
        final_df.loc[
            final_df["_CUSTOMER_ID"].duplicated(
                keep=False
            ),
            "_CUSTOMER_ID"
        ]
        .value_counts()
        .head(20)
    )

    print("\nCác CUSTOMER_NUMBER trùng nhiều nhất:")
    print(duplicated_ids)

    raise ValueError(
        "Final dataset có CUSTOMER_NUMBER trùng lặp. "
        "Không thể merge one-to-one."
    )


# ============================================================
# 10. CHỈ GIỮ KHÁCH HÀNG TRONG FINAL DATASET
# ============================================================

final_ids = set(
    final_df["_CUSTOMER_ID"]
    .dropna()
    .unique()
)


customer_sub = customer_df.loc[
    customer_df["_CUSTOMER_ID"].isin(final_ids),
    [
        "_CUSTOMER_ID",
        "_RELATIONSHIP_DATE"
    ]
].copy()


card_sub = card_df.loc[
    card_df["_CUSTOMER_ID"].isin(final_ids),
    [
        "_CUSTOMER_ID",
        "_CARD_MONTH",
        "_CARD_COUNT"
    ]
].copy()


# ============================================================
# 11. NGÀY BẮT ĐẦU QUAN HỆ KHÁCH HÀNG
# ============================================================

customer_dates = (
    customer_sub
    .dropna(
        subset=["_CUSTOMER_ID"]
    )
    .groupby(
        "_CUSTOMER_ID",
        as_index=False
    )
    .agg(
        relationship_date=(
            "_RELATIONSHIP_DATE",
            "min"
        )
    )
)


# ============================================================
# 12. NGÀY ĐẦU TIÊN QUAN SÁT THẤY THẺ TÍN DỤNG
# ============================================================

positive_card_rows = card_sub.loc[
    card_sub["_CUSTOMER_ID"].notna()
    & card_sub["_CARD_MONTH"].notna()
    & card_sub["_CARD_COUNT"].gt(0)
].copy()


first_card_dates = (
    positive_card_rows
    .groupby(
        "_CUSTOMER_ID",
        as_index=False
    )
    .agg(
        first_card_date=(
            "_CARD_MONTH",
            "min"
        )
    )
)


# ============================================================
# 13. TẠO TIMELINE
# ============================================================

timeline_df = (
    final_df[["_CUSTOMER_ID"]]
    .drop_duplicates()
    .merge(
        customer_dates,
        on="_CUSTOMER_ID",
        how="left",
        validate="one_to_one"
    )
    .merge(
        first_card_dates,
        on="_CUSTOMER_ID",
        how="left",
        validate="one_to_one"
    )
)


# ============================================================
# 14. TẠO NGÀY CẮT FLDC-60D
# ============================================================

timeline_df["feature_cutoff_date"] = (
    timeline_df["relationship_date"]
    + pd.Timedelta(days=HORIZON_DAYS)
)

timeline_df["days_to_first_card"] = (
    timeline_df["first_card_date"]
    - timeline_df["relationship_date"]
).dt.days


# ============================================================
# 15. XÁC ĐỊNH CÁC ĐIỀU KIỆN LOẠI
# ============================================================

timeline_df["missing_relationship_date"] = (
    timeline_df["relationship_date"].isna()
)


timeline_df["insufficient_60d_window"] = (
    timeline_df["relationship_date"].notna()
    & (
        timeline_df["feature_cutoff_date"]
        > FIXED_END_DATE
    )
)


timeline_df["invalid_card_date"] = (
    timeline_df["first_card_date"].notna()
    & timeline_df["relationship_date"].notna()
    & (
        timeline_df["first_card_date"]
        < timeline_df["relationship_date"]
    )
)


timeline_df["early_adopter_60d"] = (
    timeline_df["first_card_date"].notna()
    & timeline_df["relationship_date"].notna()
    & (
        timeline_df["first_card_date"]
        >= timeline_df["relationship_date"]
    )
    & (
        timeline_df["first_card_date"]
        <= timeline_df["feature_cutoff_date"]
    )
)


# ============================================================
# 16. XÁC ĐỊNH QUẦN THỂ FLDC-60D
# ============================================================

timeline_df["eligible_fldc60"] = (
    ~timeline_df["missing_relationship_date"]
    & ~timeline_df["insufficient_60d_window"]
    & ~timeline_df["invalid_card_date"]
    & ~timeline_df["early_adopter_60d"]
)


# ============================================================
# 17. GÁN NHÃN STRICTLY-FUTURE
# ============================================================

timeline_df["TARGET_FLDC_60D"] = 0


positive_mask = (
    timeline_df["eligible_fldc60"]
    & timeline_df["first_card_date"].notna()
    & (
        timeline_df["first_card_date"]
        > timeline_df["feature_cutoff_date"]
    )
    & (
        timeline_df["first_card_date"]
        <= FIXED_END_DATE
    )
)


timeline_df.loc[
    positive_mask,
    "TARGET_FLDC_60D"
] = 1


# ============================================================
# 18. TÍNH tenure_at_cutoff
#
# Nhãn 0:
#   31/12/2019 - relationship_date
#
# Nhãn 1:
#   first_card_date - relationship_date
# ============================================================

# Mặc định theo công thức của nhãn 0
timeline_df["tenure_at_cutoff"] = (
    FIXED_END_DATE
    - timeline_df["relationship_date"]
).dt.days


# Thay bằng công thức nhãn 1 cho positive
positive_tenure_mask = (
    timeline_df["eligible_fldc60"]
    & timeline_df["TARGET_FLDC_60D"].eq(1)
)


timeline_df.loc[
    positive_tenure_mask,
    "tenure_at_cutoff"
] = (
    timeline_df.loc[
        positive_tenure_mask,
        "first_card_date"
    ]
    - timeline_df.loc[
        positive_tenure_mask,
        "relationship_date"
    ]
).dt.days


timeline_df["tenure_at_cutoff"] = pd.to_numeric(
    timeline_df["tenure_at_cutoff"],
    errors="coerce"
).astype("Int64")


# ============================================================
# 19. GÁN LÝ DO LOẠI
# ============================================================

timeline_df["exclusion_reason"] = "INCLUDED_FLDC60"


timeline_df.loc[
    timeline_df["missing_relationship_date"],
    "exclusion_reason"
] = "MISSING_RELATIONSHIP_DATE"


timeline_df.loc[
    timeline_df["insufficient_60d_window"],
    "exclusion_reason"
] = "INSUFFICIENT_60D_WINDOW"


timeline_df.loc[
    timeline_df["invalid_card_date"],
    "exclusion_reason"
] = "INVALID_CARD_BEFORE_RELATIONSHIP"


timeline_df.loc[
    timeline_df["early_adopter_60d"],
    "exclusion_reason"
] = "EARLY_ADOPTER_WITHIN_60D"


# ============================================================
# 20. TẠO COHORT NHÁNH C
# ============================================================

branch_c_timeline = timeline_df.loc[
    timeline_df["eligible_fldc60"]
].copy()


branch_c_ids = set(
    branch_c_timeline["_CUSTOMER_ID"]
    .dropna()
    .unique()
)


branch_c_df = final_df.loc[
    final_df["_CUSTOMER_ID"].isin(branch_c_ids)
].copy()


# ============================================================
# 21. XÓA NHÃN CŨ VÀ GHÉP NHÃN MỚI
# ============================================================

branch_c_df.drop(
    columns=["COUNT_CREDITCARD"],
    inplace=True,
    errors="ignore"
)


branch_c_df = branch_c_df.merge(
    branch_c_timeline[
        [
            "_CUSTOMER_ID",
            "relationship_date",
            "feature_cutoff_date",
            "first_card_date",
            "days_to_first_card",
            "tenure_at_cutoff",
            "TARGET_FLDC_60D"
        ]
    ],
    on="_CUSTOMER_ID",
    how="inner",
    validate="one_to_one"
)


branch_c_df.rename(
    columns={
        "TARGET_FLDC_60D": "COUNT_CREDITCARD"
    },
    inplace=True
)


# ============================================================
# 22. TẠO CÁC BẢNG BỊ LOẠI
# ============================================================

early_adopter_df = timeline_df.loc[
    timeline_df["early_adopter_60d"]
].copy()


insufficient_window_df = timeline_df.loc[
    timeline_df["insufficient_60d_window"]
].copy()


invalid_date_df = timeline_df.loc[
    timeline_df["invalid_card_date"]
].copy()


# ============================================================
# 23. THỐNG KÊ COHORT
# ============================================================

n_initial = len(final_df)

n_missing_relationship = int(
    timeline_df["missing_relationship_date"].sum()
)

n_insufficient = int(
    timeline_df["insufficient_60d_window"].sum()
)

n_invalid = int(
    timeline_df["invalid_card_date"].sum()
)

n_early_eligible_window = int(
    (
        timeline_df["early_adopter_60d"]
        & ~timeline_df["insufficient_60d_window"]
        & ~timeline_df["missing_relationship_date"]
        & ~timeline_df["invalid_card_date"]
    ).sum()
)

n_final = len(branch_c_df)

n_positive = int(
    branch_c_df["COUNT_CREDITCARD"].sum()
)

n_negative = int(
    branch_c_df["COUNT_CREDITCARD"]
    .eq(0)
    .sum()
)

positive_rate = (
    n_positive / n_final
    if n_final > 0
    else np.nan
)


# ============================================================
# 24. KIỂM TRA TÍNH NHẤT QUÁN
# ============================================================

assert len(branch_c_df) == branch_c_df[ID_COL].nunique(), (
    "Cohort nhánh C có CUSTOMER_NUMBER trùng lặp."
)


assert branch_c_df["COUNT_CREDITCARD"].isin(
    [0, 1]
).all(), (
    "Nhãn không chỉ gồm 0 và 1."
)


assert (
    branch_c_df["feature_cutoff_date"]
    <= FIXED_END_DATE
).all(), (
    "Vẫn còn khách hàng không đủ 60 ngày quan sát."
)


assert not (
    branch_c_df["first_card_date"].notna()
    & (
        branch_c_df["first_card_date"]
        <= branch_c_df["feature_cutoff_date"]
    )
).any(), (
    "Vẫn còn khách hàng mở thẻ trong 60 ngày đầu."
)


assert n_positive + n_negative == n_final, (
    "Tổng nhãn 0 và nhãn 1 không bằng kích thước cohort."
)


# ============================================================
# 25. KIỂM TRA tenure_at_cutoff
# ============================================================

assert branch_c_df["tenure_at_cutoff"].notna().all(), (
    "Có khách hàng thiếu tenure_at_cutoff."
)


assert branch_c_df["tenure_at_cutoff"].ge(0).all(), (
    "Có tenure_at_cutoff âm."
)


assert branch_c_df["tenure_at_cutoff"].ge(
    HORIZON_DAYS
).all(), (
    "Có khách hàng có tenure_at_cutoff nhỏ hơn 60 ngày."
)


# Kiểm tra nhãn 1
positive_check = branch_c_df.loc[
    branch_c_df["COUNT_CREDITCARD"].eq(1)
].copy()


expected_positive_tenure = (
    positive_check["first_card_date"]
    - positive_check["relationship_date"]
).dt.days.astype("Int64")


actual_positive_tenure = (
    positive_check["tenure_at_cutoff"]
    .astype("Int64")
)


assert (
    actual_positive_tenure.reset_index(drop=True)
    == expected_positive_tenure.reset_index(drop=True)
).all(), (
    "tenure_at_cutoff của nhóm nhãn 1 bị sai."
)


# Kiểm tra nhãn 0
negative_check = branch_c_df.loc[
    branch_c_df["COUNT_CREDITCARD"].eq(0)
].copy()


expected_negative_tenure = (
    FIXED_END_DATE
    - negative_check["relationship_date"]
).dt.days.astype("Int64")


actual_negative_tenure = (
    negative_check["tenure_at_cutoff"]
    .astype("Int64")
)


assert (
    actual_negative_tenure.reset_index(drop=True)
    == expected_negative_tenure.reset_index(drop=True)
).all(), (
    "tenure_at_cutoff của nhóm nhãn 0 bị sai."
)


# ============================================================
# 26. THỐNG KÊ tenure_at_cutoff
# ============================================================

tenure_summary = (
    branch_c_df
    .groupby("COUNT_CREDITCARD")["tenure_at_cutoff"]
    .agg(
        Customers="count",
        Minimum="min",
        Q1=lambda x: x.quantile(0.25),
        Median="median",
        Mean="mean",
        Q3=lambda x: x.quantile(0.75),
        Maximum="max"
    )
    .round(2)
)


# ============================================================
# 27. XÓA CỘT KỸ THUẬT TRƯỚC KHI LƯU
# ============================================================

branch_c_df.drop(
    columns=["_CUSTOMER_ID"],
    inplace=True,
    errors="ignore"
)


# ============================================================
# 28. XÓA FILE ĐẦU RA CŨ
#
# Việc này giúp chạy lại rõ ràng và tránh giữ file cũ
# trong trường hợp lần ghi trước bị gián đoạn.
# ============================================================

output_files = [
    OUTPUT_PATH,
    TIMELINE_PATH,
    EARLY_ADOPTER_PATH,
    INSUFFICIENT_WINDOW_PATH,
    INVALID_DATE_PATH
]


for file_path in output_files:
    try:
        if file_path.exists():
            file_path.unlink()
            print(f"Đã xóa file cũ: {file_path.name}")
    except PermissionError:
        raise PermissionError(
            f"Không thể xóa file {file_path}. "
            "Hãy đóng file nếu đang mở trong Excel hoặc Google Drive."
        )


# ============================================================
# 29. XUẤT FILE
# ============================================================

branch_c_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)


timeline_df.to_csv(
    TIMELINE_PATH,
    index=False,
    encoding="utf-8-sig"
)


early_adopter_df.to_csv(
    EARLY_ADOPTER_PATH,
    index=False,
    encoding="utf-8-sig"
)


insufficient_window_df.to_csv(
    INSUFFICIENT_WINDOW_PATH,
    index=False,
    encoding="utf-8-sig"
)


invalid_date_df.to_csv(
    INVALID_DATE_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 30. KIỂM TRA FILE SAU KHI GHI
# ============================================================

for file_path in output_files:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Không tạo được file đầu ra: {file_path}"
        )


# ============================================================
# 31. IN KẾT QUẢ
# ============================================================

print("\n" + "=" * 78)
print("KẾT QUẢ TẠO DỮ LIỆU E1 — NHÁNH C")
print("=" * 78)

print(f"Ngày kết thúc nghiên cứu       : {FIXED_END_DATE.date()}")
print(f"Cửa sổ FLDC                    : {HORIZON_DAYS} ngày")


print("\n--- QUÁ TRÌNH LỌC ---")

print(
    f"Số khách hàng ban đầu          : "
    f"{n_initial:,}"
)

print(
    f"Thiếu ngày bắt đầu quan hệ     : "
    f"{n_missing_relationship:,}"
)

print(
    f"Không đủ 60 ngày quan sát      : "
    f"{n_insufficient:,}"
)

print(
    f"Ngày thẻ trước ngày quan hệ    : "
    f"{n_invalid:,}"
)

print(
    f"Mở thẻ trong 60 ngày đầu       : "
    f"{n_early_eligible_window:,}"
)

print(
    f"Số khách hàng nhánh C          : "
    f"{n_final:,}"
)


print("\n--- PHÂN BỐ NHÃN ---")

print(
    f"Negative – nhãn 0              : "
    f"{n_negative:,}"
)

print(
    f"Positive – nhãn 1              : "
    f"{n_positive:,}"
)

print(
    f"Positive rate                  : "
    f"{positive_rate:.4%}"
)


label_statistics = (
    branch_c_df["COUNT_CREDITCARD"]
    .value_counts()
    .sort_index()
    .rename_axis("Label")
    .reset_index(name="Customers")
)


label_statistics["Percentage"] = (
    label_statistics["Customers"]
    / n_final
    * 100
)


print("\nBảng tần số nhãn:")

print(
    label_statistics.to_string(
        index=False
    )
)


print("\n--- THỐNG KÊ tenure_at_cutoff ---")

print(
    tenure_summary.to_string()
)


print("\n--- CẤU TRÚC FILE NHÁNH C ---")

print(
    f"Số dòng: {branch_c_df.shape[0]:,}"
)

print(
    f"Số cột: {branch_c_df.shape[1]:,}"
)

print("\nCác cột:")

print(
    branch_c_df.columns.tolist()
)


print("\n--- FILE ĐÃ TẠO ---")

print(f"1. Dữ liệu nhánh C:\n   {OUTPUT_PATH}")

print(f"\n2. Timeline:\n   {TIMELINE_PATH}")

print(
    "\n3. Early adopters:\n"
    f"   {EARLY_ADOPTER_PATH}"
)

print(
    "\n4. Không đủ cửa sổ 60 ngày:\n"
    f"   {INSUFFICIENT_WINDOW_PATH}"
)

print(
    "\n5. Ngày bất thường:\n"
    f"   {INVALID_DATE_PATH}"
)


print("\nKIỂM TRA ĐẠT:")

print("- Code có thể chạy lại nhiều lần.")
print("- Google Drive không bị mount lặp.")
print("- File nguồn được tự động tìm.")
print("- File đầu ra cũ được ghi đè.")
print("- Quần thể là FLDC-60D.")
print("- Nhãn 1 là adoption sau ngày thứ 60.")
print("- Nhãn 0 dùng mốc 31/12/2019.")
print("- Các feature ELAPS cũ được giữ nguyên.")
print("- tenure_at_cutoff đã được bổ sung.")
print("- Hoàn thành tạo file nhánh C.")

Mounted at /content/drive
Đã dọn dữ liệu còn lại từ lần chạy trước.

TÌM THẤY 2 FILE TRÙNG TÊN: final_dataset_no_auto_job.csv
1. /content/drive/MyDrive/ELAPSPLACEBO/final_dataset_no_auto_job.csv
2. /content/drive/MyDrive/Fintect/final_dataset_no_auto_job.csv

Mặc định sử dụng file đầu tiên:
/content/drive/MyDrive/ELAPSPLACEBO/final_dataset_no_auto_job.csv

TÌM THẤY 2 FILE TRÙNG TÊN: 1.Data_Customer.csv
1. /content/drive/MyDrive/ELAPSPLACEBO/1.Data_Customer.csv
2. /content/drive/MyDrive/Fintect/1.Data_Customer.csv

Mặc định sử dụng file đầu tiên:
/content/drive/MyDrive/ELAPSPLACEBO/1.Data_Customer.csv

TÌM THẤY 2 FILE TRÙNG TÊN: 6.Data_Card.xlsx
1. /content/drive/MyDrive/ELAPSPLACEBO/6.Data_Card.xlsx
2. /content/drive/MyDrive/Fintect/6.Data_Card.xlsx

Mặc định sử dụng file đầu tiên:
/content/drive/MyDrive/ELAPSPLACEBO/6.Data_Card.xlsx

ĐƯỜNG DẪN ĐƯỢC SỬ DỤNG
Final dataset : /content/drive/MyDrive/ELAPSPLACEBO/final_dataset_no_auto_job.csv
Customer      : /content/drive/MyDrive/ELAPSPLAC